In [ ]:
pip install spacy

   ---------------------------------------- 0.0/15.2 MB ? eta -:--:--
    --------------------------------------- 0.3/15.2 MB ? eta -:--:--
   -- ------------------------------------- 1.0/15.2 MB 2.8 MB/s eta 0:00:06
   ------- -------------------------------- 2.9/15.2 MB 6.1 MB/s eta 0:00:03
   --------------- ------------------------ 5.8/15.2 MB 8.4 MB/s eta 0:00:02
   ----------------- ---------------------- 6.6/15.2 MB 7.8 MB/s eta 0:00:02
   --------------------- ------------------ 8.1/15.2 MB 7.2 MB/s eta 0:00:01
   -------------------------------- ------- 12.3/15.2 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 15.2/15.2 MB 10.2 MB/s  0:00:01
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----------------------------------- ---- 1.8/2.0 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 9.8 MB/s  0:00:00
   ---------------------------------------- 0.0/650.8 kB ? eta -:--:--
   -----------------------------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import sys
!{sys.executable} -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta 


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
import spacy

In [7]:
nltk.download('stopwords')
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])  # keep only tagger+lemmatizer, faster

df = pd.read_parquet('DATASET.parquet')
print(df.shape)
print(df['label'].value_counts().sort_index())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dyash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


(25000, 2)
label
0    3614
1    3357
2    2099
3     204
4     705
5    1730
6    5408
7    5321
8    2562
Name: count, dtype: int64


In [8]:
def basic_clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s\.\,\-]', ' ', text)   # keep basic punctuation, strip weird symbols
    text = re.sub(r'\s+', ' ', text).strip()          # collapse whitespace
    return text

df['text_clean'] = df['text'].apply(basic_clean)

In [9]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join(w for w in text.split() if w not in stop_words)

df['text_clean'] = df['text_clean'].apply(remove_stopwords)

In [10]:
def lemmatize_batch(texts, batch_size=64):
    lemmatized = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        lemmatized.append(' '.join(tok.lemma_ for tok in doc if not tok.is_space))
    return lemmatized


In [11]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=42
)
print(train_df['label'].value_counts().sort_index())
print(val_df['label'].value_counts().sort_index())

label
0    3072
1    2853
2    1784
3     173
4     599
5    1471
6    4597
7    4523
8    2178
Name: count, dtype: int64
label
0    542
1    504
2    315
3     31
4    106
5    259
6    811
7    798
8    384
Name: count, dtype: int64


In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))

# fit ONLY on train, then transform both
X_train = vectorizer.fit_transform(train_df['text_clean'])
X_val = vectorizer.transform(val_df['text_clean'])

# save it so SVM training script can just load it later
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

NameError: name 'train_df' is not defined